Load ratings into the same four-column schema.
Create one fixed train/test split.
Check cold-start coverage.
Save the split as Parquet.
Repeat core scaling with 1, 2, 4 cores.
Repeat dataset-size scaling with larger subsets.
Compare 100K versus 1M.

In [2]:
import json
import os
import sys
import urllib.request
import zipfile
from pathlib import Path
from time import perf_counter



os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ML1M_DIR = DATA_DIR / "ml-1m"
RATINGS_PATH = ML1M_DIR / "ratings.dat"
ZIP_PATH = DATA_DIR / "ml-1m.zip"

PROCESSED_DIR = DATA_DIR / "processed" / "ml-1m"
TRAIN_PARQUET = PROCESSED_DIR / "train"
TEST_PARQUET = PROCESSED_DIR / "test"

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Ratings path:", RATINGS_PATH)



Project root: /home/anna/projects/ccdpp-pyspark-movielens
Python: /home/anna/projects/ccdpp-pyspark-movielens/.venv/bin/python
Ratings path: /home/anna/projects/ccdpp-pyspark-movielens/data/ml-1m/ratings.dat


In [3]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not RATINGS_PATH.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    if not ZIP_PATH.exists():
        print("Downloading MovieLens 1M...")
        urllib.request.urlretrieve(ML1M_URL, ZIP_PATH)

    print("Extracting MovieLens 1M...")
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(DATA_DIR)

if not RATINGS_PATH.exists():
    raise FileNotFoundError(
        f"MovieLens ratings file was not found at {RATINGS_PATH}"
    )

print("MovieLens 1M is ready:", RATINGS_PATH)

MovieLens 1M is ready: /home/anna/projects/ccdpp-pyspark-movielens/data/ml-1m/ratings.dat


In [4]:
def stop_active_spark() -> None:
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()


def create_spark_session(num_cores) -> SparkSession:
    spark = (
        SparkSession.builder
        .master(f"local[{num_cores}]")
        .appName(f"als_ml1m_{num_cores}_cores")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .config(
            "spark.driver.extraJavaOptions",
            f"-Dhadoop.home.dir={hadoop_home}"
        )
        .getOrCreate()
    )
    return spark

stop_active_spark()
spark = create_spark_session(4)

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/30 19:47:35 WARN Utils: Your hostname, DESKTOP-MU43GAL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 19:47:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/anna/projects/ccdpp-pyspark-movielens/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/30 19:47:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
Default parallelism: 4
Spark UI: http://10.255.255.254:4040


In [6]:
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

print(
    "Java hadoop.home.dir:",
    spark.sparkContext._jvm.java.lang.System.getProperty(
        "hadoop.home.dir"
    )
)



HADOOP_HOME: /home/anna/projects/ccdpp-pyspark-movielens/.venv/hadoop
Java hadoop.home.dir: /home/anna/projects/ccdpp-pyspark-movielens/.venv/hadoop


In [8]:
ratings = (
    spark.read
    .schema(
        "user_id INT, "
        "item_id INT, "
        "rating DOUBLE, "
        "timestamp LONG"
    )
    .csv(
        "/home/anna/projects/ccdpp-pyspark-movielens/data/ml-1m/ratings.dat",
        sep="::",
    )
)
num_ratings = ratings.count()

ratings.show(5)
ratings.printSchema()
print("Ratings:", num_ratings)
print("Input partitions:", ratings.rdd.getNumPartitions())

+-------+-------+------+---------+
|user_id|item_id|rating|timestamp|
+-------+-------+------+---------+
|      1|   1193|   5.0|978300760|
|      1|    661|   3.0|978302109|
|      1|    914|   3.0|978301968|
|      1|   3408|   4.0|978300275|
|      1|   2355|   5.0|978824291|
+-------+-------+------+---------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)

Ratings: 1000209
Input partitions: 4


In [9]:
ratings_with_bucket = ratings.withColumn(
    "split_bucket",
    F.pmod(
        F.xxhash64("user_id", "item_id", "timestamp"),
        F.lit(10),
    )
)
ratings_with_bucket.printSchema()
ratings_with_bucket.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- split_bucket: long (nullable = true)

+-------+-------+------+---------+------------+
|user_id|item_id|rating|timestamp|split_bucket|
+-------+-------+------+---------+------------+
|      1|   1193|   5.0|978300760|           9|
|      1|    661|   3.0|978302109|           3|
|      1|    914|   3.0|978301968|           0|
|      1|   3408|   4.0|978300275|           1|
|      1|   2355|   5.0|978824291|           5|
+-------+-------+------+---------+------------+
only showing top 5 rows


In [10]:
train = (ratings_with_bucket
         .filter(F.col("split_bucket")!=9)
         )
      
test = (ratings_with_bucket
         .filter(F.col("split_bucket")==9)
         )

train.show(5)

+-------+-------+------+---------+------------+
|user_id|item_id|rating|timestamp|split_bucket|
+-------+-------+------+---------+------------+
|      1|    661|   3.0|978302109|           3|
|      1|    914|   3.0|978301968|           0|
|      1|   3408|   4.0|978300275|           1|
|      1|   2355|   5.0|978824291|           5|
|      1|   1197|   3.0|978302268|           1|
+-------+-------+------+---------+------------+
only showing top 5 rows


In [11]:
train = (ratings_with_bucket
         .filter(F.col("split_bucket")!=9)
         .drop("split_bucket")
         )
      
test = (ratings_with_bucket
         .filter(F.col("split_bucket")==9)
         .drop("split_bucket")
         )
train.show(5)

+-------+-------+------+---------+
|user_id|item_id|rating|timestamp|
+-------+-------+------+---------+
|      1|    661|   3.0|978302109|
|      1|    914|   3.0|978301968|
|      1|   3408|   4.0|978300275|
|      1|   2355|   5.0|978824291|
|      1|   1197|   3.0|978302268|
+-------+-------+------+---------+
only showing top 5 rows


In [12]:
print("train:", train.count())
print("test:", test.count())
print("total:", train.count() + test.count())

train: 900560


test: 99649


total: 1000209


In [13]:
train_users = train.select("user_id").distinct()
test_users = test.select("user_id").distinct()

train_items = train.select("item_id").distinct()
test_items = test.select("item_id").distinct()

unseen_users = test_users.join(train_users, "user_id", "left_anti")
unseen_items = test_items.join(train_items, "item_id", "left_anti")

print("unseen movies: ", unseen_items.select("item_id").distinct().count())
print(unseen_users.select("user_id").distinct().count())

unseen movies:  13


0


In [14]:
test_clean = (
    test
    .join(train_users, "user_id", "left_semi")
    .join(train_items, "item_id", "left_semi")
)

In [15]:
print("test-only users:", unseen_users.count())
print("test-only items:", unseen_items.count())
print("original test rows:", test.count())
print("clean test rows:", test_clean.count())
print("coverage:", test_clean.count() / test.count())

test-only users: 0


test-only items: 13


original test rows: 99649


clean test rows: 99636


coverage: 0.9998695420927456


In [16]:
train.write.mode("overwrite").parquet("data/processed/ml-1m/train")
test_clean.write.mode("overwrite").parquet("data/processed/ml-1m/test_clean")